In [ ]:
pip install pandas


In [ ]:
pip install comut

In [ ]:
import pandas as pd
from comut import comut, fileparsers

GBM_high_sig = ["PTEN", "TP53", "EGFR", "PIK3R1", "PIK3CA", "NF1", "RB1", "IDH1", "STAG2","RPL5", "SLC26A3",
                "CHD8", "BRAF", "MAP3K1", "QKI", "SETD2", "CD1D", "BCOR", "SMC1A", "PTPN11"]

coding_var_types = ["Missense_Mutation", "Frame_Shift_Ins", "Frame_Shift_Del", "Nonstop_Mutation", 
					"Splv2_Site", "In_Frame_Ins", "In_Frame_Del", "Nonsense_Mutation"]

In [ ]:
def load_mut_tables(v2_tsv, v6_tsv, var_types = None):
    v2_table = pd.read_csv(v2_tsv, sep="\t")
    v6_table = pd.read_csv(v6_tsv, sep="\t")
    root_sample_dict = dict()
    root_sample_dict["sample"] = []
    root_sample_dict["root_sample"] = v6_table["sample"].to_list()
    root_sample_dict["root_sample"].extend(v2_table["sample"].to_list())

    v6_table["sample"] = [s + "_wes_v6" for s in v6_table["sample"].to_list()]
    v2_table["sample"] = [s + "_wes_v2" for s in v2_table["sample"].to_list()]
    mut_data = pd.concat([v6_table, v2_table])
    mut_data = mut_data.sort_values(by=["sample"], ignore_index=True)
    if var_types is not None:
        mut_data = mut_data[[m in var_types for m in mut_data['value'].to_list()]]

    root_sample_dict["sample"].extend(v6_table["sample"].to_list())
    root_sample_dict["sample"].extend(v2_table["sample"].to_list())
    root_sample_df = pd.DataFrame.from_dict(root_sample_dict)
    root_sample_df = root_sample_df.sort_values(by=["sample"], ignore_index=True).drop_duplicates(ignore_index=True)
    root_sample_df["group"] = sorted([n for n in range(0, int(len(root_sample_df.index)/2))] * 2)

    platform_dict = dict()
    platform_dict["sample"] = root_sample_df["sample"].to_list()
    platform_dict["category"] = ["Platform"] * len(platform_dict["sample"])
    platform_dict["value"] = ["WES_V2", "WES_V6"] * int(len(root_sample_df.index)/2)
    platform_df = pd.DataFrame.from_dict(platform_dict)
    return mut_data, root_sample_df, platform_df


def plot_comut(mut_data, root_sample_df, platform_df):
    gbm_comut = comut.CoMut()
    gbm_comut.samples = root_sample_df["sample"].to_list()
    gbm_comut.add_sample_indicators(root_sample_df, name="Root Sample", 
                                    plot_kwargs={'color': 'black', 'marker': '|', 'markersize': 12})
    gbm_comut.add_categorical_data(platform_df, name="Platform")
    gbm_comut.add_categorical_data(mut_data, name="Mutation Type", category_order=[g for g in GBM_high_sig if g in mut_data["category"].to_list()][::-1])
    plot_struct = [["Root Sample","Platform"], ["Mutation Type"]]
    gbm_comut.plot_comut(figsize=(16, 7), x_padding=0.04, y_padding=0.04, tri_padding=0.03,
                         structure=plot_struct, subplot_hspace=-0.35)
    gbm_comut.add_unified_legend()
    return gbm_comut

In [ ]:
gbm_full_mut, root_sample_df, platform_df = load_mut_tables("v2_comut_data.var_index.tsv","v6_comut_data.var_index.tsv")
gbm_full_mut = gbm_full_mut[[m in GBM_high_sig for m in gbm_full_mut["category"].to_list()]]
for idx, row in gbm_full_mut.iterrows():
	gbm_full_mut.at[idx, "var_index"] = "_".join([str(row['category']),str(row['chr']), str(row['start']), str(row['end']),
													str(row['ref']), str(row['alt']), str(row['sample']).replace('_wes_v2','').replace('_wes_v6','')])
	gbm_full_mut.at[idx, "platform"] = str(row['sample'])[-6:]

In [ ]:
var_count_df = dict()
gbm_full_mut = gbm_full_mut[[g in coding_var_types for g in gbm_full_mut['value']]]
present_genes = list(set(gbm_full_mut['category'].to_list()))
var_categ_df = {'sample': [],'category': [], 'value': []}
for g in present_genes:
	# Columns will be  shared, v2_only, v6_only
	var_count_df[g] = [0,0,0]
for var in list(set(gbm_full_mut['var_index'].to_list())):
	gene = var.split("_")[0]
	present_in = list(set(gbm_full_mut.loc[gbm_full_mut['var_index']==var,'platform'].to_list()))
	var_categ_df['category'].append(gene)
	var_categ_df['sample'].append("_".join(var.split("_")[6:]) + "_wes_v2")
	if len(present_in) == 2:
		var_count_df[gene][0]+=1
		var_categ_df['value'].append("Shared")
	elif present_in[0] == "wes_v2":
		var_count_df[gene][1]+=1
		var_categ_df['value'].append("WES_V2")
	elif present_in[0] == "wes_v6":
		var_count_df[gene][2]+=1
		var_categ_df['value'].append("WES_V6")
	else:
		print(present_in)
		var_categ_df['value'].append("unknown")
var_count_df = pd.DataFrame.from_dict(var_count_df, orient='index', columns=['shared','v2_only','v6_only'])
var_count_df = var_count_df.reset_index(names="category")
var_categ_df = pd.DataFrame.from_dict(var_categ_df)
var_count_df

In [ ]:
gbm_full_mut.head()

In [ ]:
gene_var_counts = dict()
for idx,row in var_count_df.iterrows():
	gene_var_counts[row['category']] = row['shared'] + row['v2_only'] + row['v6_only']
gene_var_counts = pd.DataFrame.from_dict(gene_var_counts, orient='index', columns=['coding_vars'])
gene_var_counts

In [ ]:
# Initial plot to add to
gbm_comut = plot_comut(gbm_full_mut, root_sample_df, platform_df)

In [ ]:
var_count_df['category'] = var_count_df['category'].to_list()

In [ ]:
gbm_comut.add_categorical_data(var_categ_df, name='Coding Vars', category_order=['Shared', 'WES_V2', 'WES_V6'],
							   mapping={'WES_V2': 'limegreen', 'WES_V6': 'tomato', 'Shared': 'royalblue'})
gbm_comut.add_side_bar_data(var_count_df, paired_name='Mutation Type', name = "Coding Vars",
							position='right', bar_kwargs={'height': 0.8}, xlabel="Coding Vars",
							mapping={'v2_only': 'limegreen', 'v6_only': 'tomato', 'shared': 'royalblue'},
							stacked=True)

In [ ]:
plot_struct = [["Root Sample","Platform"], ["Mutation Type"]]
gbm_comut.plot_comut(figsize=(12, 7), x_padding=0.04, y_padding=0.04, tri_padding=0.03,
                         structure=plot_struct, subplot_hspace=-0.35, wspace=0.03)
gbm_comut.add_unified_legend(axis_name='Coding Vars')

In [ ]:
# Final Plot
gbm_comut.figure.set_size_inches(15.5,9.0)
gbm_comut.figure.get_figure()